# ARC-v0.9 — Boundary & Stability Map

Post-confirmatory study of **when approximation-feedback amplification occurs, when it is bounded, and why**.

This notebook keeps positive, null, and reversal cases. It creates a new deterministic 50/50
boundary-fit / boundary-validation split before the boundary sweep.

Main analyses:

1. **Fidelity dose response:** PQ32↔PQ64, PQ64↔SQ8, PQ32↔SQ8.
2. **Feedback-gain boundary:** alpha / k / softmax temperature.
3. **Query-level prediction:** can initial retrieval geometry predict later H3 amplification?
4. **Stable / amplifying / reversal map:** validated out of sample.

ARC-v0.8 remains the sealed confirmation. ARC-v0.9 is explicitly post-confirmatory.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, gc, sqlite3, time
import numpy as np
import pandas as pd
import faiss
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score, average_precision_score, accuracy_score

print("FAISS:", faiss.__version__)
print("RAM GB:", psutil.virtual_memory().total / 1024**3)


## 1. Paths, provenance, and boundary-study seal


In [ ]:
SEED = 20260816
DIM = 384
N_DOCS = 5_233_329
NPROBE = 64
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

ROOT = Path("/content/drive/MyDrive/hc-rars-external-confirmation-hotpotqa-5m-v1")
ARC_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0")
INDEX_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/hotpotqa-rebuilt-v3")

SHARD_ROOT = ROOT / "stage1/corpus-embedding-shards-v3"
SHARD_MANIFEST = SHARD_ROOT / "manifest.json"
QUERY_EMB = ROOT / "stage1/query_embeddings.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
CORPUS_DB = ROOT / "stage1/corpus_ids.sqlite"
DEV_QRELS = ROOT / "source/hotpotqa/qrels/dev.tsv"

INDEX_PATHS = {
    "PQ32": INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss",
    "PQ64": INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m64-nbits8-seed20260816.faiss",
    "SQ8": INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfsq8-nlist4096-seed20260816.faiss",
}

V08_RUNS = sorted(
    [p for p in (ARC_ROOT / "sealed-hotpotqa-h1-h4-confirmation-v08").glob("*") if (p / "report.json").is_file()],
    key=lambda p: p.stat().st_mtime,
)
if not V08_RUNS:
    raise FileNotFoundError("Completed ARC-v0.8 report not found.")

V08_RUN = V08_RUNS[-1]
V08_REPORT = V08_RUN / "report.json"

with open(V08_REPORT, "r", encoding="utf-8") as f:
    v08 = json.load(f)

assert v08["status"] == "SEALED_HOTPOTQA_H1_H4_CONFIRMATION_COMPLETE"
assert v08["test_retrieval_performed"] is False
assert v08["test_qrels_accessed"] is False

OUT_ROOT = ARC_ROOT / "boundary-stability-map-v09"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

for p in [SHARD_MANIFEST, QUERY_EMB, QUERY_IDS, SPLIT_MANIFEST, CORPUS_DB, DEV_QRELS, *INDEX_PATHS.values()]:
    if not p.is_file():
        raise FileNotFoundError(p)

def sha256_file(path, chunk_size=64*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

protocol = {
    "status": "POST_CONFIRMATORY_BOUNDARY_STUDY_SEALED_BEFORE_SWEEP",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_v08_report": str(V08_REPORT),
    "source_v08_report_sha256": sha256_file(V08_REPORT),
    "query_split": "deterministic SHA256 parity 50/50 fit-validation",
    "fidelity_pairs": [["PQ32","PQ64"],["PQ64","SQ8"],["PQ32","SQ8"]],
    "rounds": MAX_ROUNDS,
    "retain_null_and_reversal_cases": True,
    "test_access_allowed": False,
}

raw = json.dumps(protocol, sort_keys=True, separators=(",",":")).encode()
protocol_sha = hashlib.sha256(raw).hexdigest()
protocol["protocol_sha256"] = protocol_sha

(OUT / "boundary_protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")
print("v0.8:", V08_RUN)
print("v0.9:", OUT)
print("protocol SHA:", protocol_sha)


## 2. Load rebuilt shards, DEV queries, and relevance


In [ ]:
with open(SHARD_MANIFEST, "r", encoding="utf-8") as f:
    shard_manifest = json.load(f)

assert shard_manifest["status"] == "CORPUS_EMBEDDING_SHARDS_COMPLETE"
shards = sorted(shard_manifest["shards"], key=lambda x: int(x["shard_id"]))
SHARD_ENDS = np.asarray([int(s["end_row"]) for s in shards], dtype=np.int64)

queries = np.load(QUERY_EMB, mmap_mode="r")
with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x) for x in split["dev_query_ids"]]
assert len(dev_ids) == 5447
assert split["test_qrels_relevance_values_accessed"] is False
assert split["test_retrieval_performed"] is False
assert split["test_outcomes_observed"] is False

query_row = {qid:i for i,qid in enumerate(query_ids)}
dev_query_rows = np.asarray([query_row[q] for q in dev_ids], dtype=np.int64)

Q_DEV = np.asarray(queries[dev_query_rows], dtype=np.float32)
Q_DEV /= np.maximum(np.linalg.norm(Q_DEV, axis=1, keepdims=True), 1e-12)

dev = pd.read_csv(DEV_QRELS, sep="\t")
dev["query-id"] = dev["query-id"].astype(str)
dev["corpus-id"] = dev["corpus-id"].astype(str)

unique_doc_ids = dev["corpus-id"].drop_duplicates().tolist()

with sqlite3.connect(str(CORPUS_DB)) as con:
    con.execute("CREATE TEMP TABLE requested_ids (doc_id TEXT PRIMARY KEY)")
    con.executemany("INSERT INTO requested_ids(doc_id) VALUES (?)", [(x,) for x in unique_doc_ids])
    mapped = pd.read_sql_query(
        "SELECT r.doc_id, d.row_id FROM requested_ids r LEFT JOIN documents d ON d.doc_id=r.doc_id",
        con,
    )

assert mapped["row_id"].notna().all()
mapped["row_id"] = mapped["row_id"].astype(np.int64)

dev = dev.merge(mapped, left_on="corpus-id", right_on="doc_id", how="left", validate="many_to_one")
dev = dev.rename(columns={"row_id":"corpus-row"})
dev["corpus-row"] = dev["corpus-row"].astype(np.int64)

dev_qrels = {
    str(qid): set(g.loc[g["score"]>0, "corpus-row"].astype(np.int64).tolist())
    for qid,g in dev.groupby("query-id")
}
assert len(dev_qrels) == 5447

print("DEV alignment — PASS")


## 3. Freeze boundary-fit / boundary-validation split


In [ ]:
def split_bucket(qid):
    h = hashlib.sha256(str(qid).encode("utf-8")).digest()
    return int.from_bytes(h[:8], "big") % 2

boundary_split = pd.DataFrame({
    "query_id": dev_ids,
    "split": ["fit" if split_bucket(q)==0 else "validation" for q in dev_ids],
})

boundary_split.to_csv(OUT / "boundary_query_split.csv", index=False)

fit_mask = boundary_split["split"].to_numpy() == "fit"
val_mask = ~fit_mask

print(boundary_split["split"].value_counts())
print("BOUNDARY SPLIT — FROZEN")


## 4. Helpers


In [ ]:
def load_rows_from_shards(rows):
    rows = np.asarray(rows, dtype=np.int64)
    shape = rows.shape
    flat = rows.reshape(-1)

    unique_rows, inverse = np.unique(flat, return_inverse=True)
    shard_ids = np.searchsorted(SHARD_ENDS, unique_rows, side="right")

    vec = np.empty((len(unique_rows), DIM), dtype=np.float32)

    for sid in np.unique(shard_ids):
        mask = shard_ids == sid
        start = int(shards[int(sid)]["start_row"])
        local = unique_rows[mask] - start
        arr = np.load(SHARD_ROOT / shards[int(sid)]["file"], mmap_mode="r")
        vec[mask] = np.asarray(arr[local], dtype=np.float32)

    return vec[inverse].reshape(*shape, DIM)

def normalize_rows(x):
    x = np.asarray(x, np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

def feedback_matrix(ids, scores, cfg):
    k = int(cfg["k"])
    x = load_rows_from_shards(np.asarray(ids[:, :k], np.int64))

    if cfg["method"] == "mean":
        f = x.mean(axis=1)
    else:
        z = np.asarray(scores[:, :k], np.float64) / float(cfg["temperature"])
        z -= z.max(axis=1, keepdims=True)
        w = np.exp(np.clip(z, -60, 60))
        w /= np.maximum(w.sum(axis=1, keepdims=True), 1e-12)
        f = (x * w[:,:,None]).sum(axis=1)

    return normalize_rows(f)

def anchored_update(q0, f, alpha):
    return normalize_rows((1-float(alpha))*q0 + float(alpha)*f)

def jaccard_distance_rows(a, b):
    out = np.empty(len(a), np.float32)
    for i in range(len(a)):
        A, B = set(map(int,a[i])), set(map(int,b[i]))
        out[i] = 1.0 - len(A & B)/max(len(A | B),1)
    return out

def cosine_distance_rows(a, b):
    a, b = normalize_rows(a), normalize_rows(b)
    return 1.0 - np.sum(a*b, axis=1)

def ndcg_at_10(qids, ids):
    discounts = 1.0 / np.log2(np.arange(2, 12))
    out = np.zeros(len(qids), np.float32)
    for i,qid in enumerate(qids):
        rel = dev_qrels[str(qid)]
        hits = np.asarray([1.0 if int(d) in rel else 0.0 for d in ids[i,:10]], np.float32)
        dcg = float((hits*discounts).sum())
        ideal = min(len(rel),10)
        idcg = float(discounts[:ideal].sum())
        out[i] = dcg/idcg if idcg else 0.0
    return out

def score_entropy(scores, k=20):
    z = np.asarray(scores[:,:k], np.float64)
    z -= z.max(axis=1, keepdims=True)
    p = np.exp(np.clip(z,-60,60))
    p /= np.maximum(p.sum(axis=1, keepdims=True),1e-12)
    return -np.sum(p*np.log(np.maximum(p,1e-12)), axis=1)

def score_margin(scores):
    return np.asarray(scores[:,0]-scores[:,9], np.float32)

def cfg_key(cfg):
    temp = "none" if cfg["temperature"] is None else str(cfg["temperature"]).replace(".","p")
    return f"{cfg['method']}-k{cfg['k']}-a{str(cfg['alpha']).replace('.','p')}-t{temp}"


## 5. Fidelity pairs and boundary grid


In [ ]:
FIDELITY_PAIRS = [("PQ32","PQ64"), ("PQ64","SQ8"), ("PQ32","SQ8")]

FROZEN_CONFIGS = [
    {"method":"mean","k":20,"alpha":0.3,"temperature":None},
    {"method":"softmax","k":5,"alpha":0.5,"temperature":0.1},
]

BOUNDARY_CONFIGS = []

for alpha in [0.1,0.3,0.5,0.7]:
    for k in [5,20,50]:
        BOUNDARY_CONFIGS.append({"method":"mean","k":k,"alpha":alpha,"temperature":None})

for alpha in [0.1,0.3,0.5,0.7]:
    for k in [5,20]:
        for temp in [0.05,0.1,0.2,0.5]:
            BOUNDARY_CONFIGS.append({"method":"softmax","k":k,"alpha":alpha,"temperature":temp})

print("Boundary configs:", len(BOUNDARY_CONFIGS))


## 6. Baseline retrieval features


In [ ]:
baseline_cache = {}

for name,path in INDEX_PATHS.items():
    print("baseline:", name)
    index = faiss.read_index(str(path))
    index.nprobe = NPROBE

    scores, ids = index.search(np.ascontiguousarray(Q_DEV,np.float32), TOP_RETRIEVE)

    baseline_cache[name] = {
        "scores": scores,
        "ids": ids,
        "ndcg": ndcg_at_10(dev_ids, ids),
        "entropy20": score_entropy(scores,20),
        "margin1_10": score_margin(scores),
    }

    print("nDCG@10:", float(baseline_cache[name]["ndcg"].mean()))
    del index
    gc.collect()

print("BASELINE CACHE — COMPLETE")


## 7. Generic paired trajectory


In [ ]:
def run_pair_trajectory(low_name, high_name, cfg, query_mask=None):
    idx = np.arange(len(dev_ids)) if query_mask is None else np.flatnonzero(query_mask)

    qids = [dev_ids[i] for i in idx]
    q0 = Q_DEV[idx].copy()

    low = faiss.read_index(str(INDEX_PATHS[low_name]))
    high = faiss.read_index(str(INDEX_PATHS[high_name]))
    low.nprobe = high.nprobe = NPROBE

    qL, qH = q0.copy(), q0.copy()
    frames = []
    base_cdiv = None

    for t in range(MAX_ROUNDS+1):
        sL,idL = low.search(np.ascontiguousarray(qL,np.float32), TOP_RETRIEVE)
        sH,idH = high.search(np.ascontiguousarray(qH,np.float32), TOP_RETRIEVE)

        qdiv = cosine_distance_rows(qL,qH)
        cdiv = jaccard_distance_rows(idL,idH)

        if t == 0:
            base_cdiv = cdiv.copy()

        nL = ndcg_at_10(qids,idL)
        nH = ndcg_at_10(qids,idH)

        frames.append(pd.DataFrame({
            "query_id": qids,
            "iteration": t,
            "query_divergence": qdiv,
            "candidate_increment": cdiv-base_cdiv,
            "abs_utility_gap": np.abs(nH-nL),
        }))

        if t < MAX_ROUNDS:
            qL = anchored_update(q0, feedback_matrix(idL,sL,cfg), cfg["alpha"])
            qH = anchored_update(q0, feedback_matrix(idH,sH,cfg), cfg["alpha"])

    del low,high
    gc.collect()

    df = pd.concat(frames,ignore_index=True)
    df["low"] = low_name
    df["high"] = high_name
    df["method"] = cfg["method"]
    df["alpha"] = cfg["alpha"]
    df["k"] = cfg["k"]
    df["temperature"] = cfg["temperature"]
    df["config_key"] = cfg_key(cfg)
    return df

def slopes_from_trajectory(df):
    rows = []

    for keys,g in df.groupby(
        ["query_id","low","high","method","alpha","k","temperature","config_key"],
        dropna=False
    ):
        qid,low,high,method,alpha,k,temp,cfgk = keys
        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)

        def slope(col):
            return float(np.polyfit(x,g[col].to_numpy(np.float64),1)[0])

        rows.append({
            "query_id":qid,
            "low":low,
            "high":high,
            "method":method,
            "alpha":alpha,
            "k":k,
            "temperature":temp,
            "config_key":cfgk,
            "H1_slope":slope("query_divergence"),
            "H2_slope":slope("candidate_increment"),
            "H3_slope":slope("abs_utility_gap"),
        })

    return pd.DataFrame(rows)


## 8. Fidelity dose response — all DEV, frozen configs


In [ ]:
dose_frames = []

for low,high in FIDELITY_PAIRS:
    for cfg in FROZEN_CONFIGS:
        print("DOSE", low, high, cfg_key(cfg))
        path = OUT / f"dose-{low.lower()}-{high.lower()}-{cfg_key(cfg)}.parquet"

        if path.is_file():
            df = pd.read_parquet(path)
        else:
            df = run_pair_trajectory(low,high,cfg)
            df.to_parquet(path,index=False)

        dose_frames.append(df)

dose_df = pd.concat(dose_frames,ignore_index=True)
dose_slopes = slopes_from_trajectory(dose_df)

dose_summary = (
    dose_slopes
    .groupby(["low","high","method"],as_index=False)[["H1_slope","H2_slope","H3_slope"]]
    .mean()
)

display(dose_summary)
dose_summary.to_csv(OUT/"fidelity_dose_response_summary.csv",index=False)
dose_slopes.to_parquet(OUT/"fidelity_dose_query_slopes.parquet",index=False)


## 9. Boundary sweep — fit half only


In [ ]:
fit_frames = []

for i,cfg in enumerate(BOUNDARY_CONFIGS,1):
    print(f"[FIT {i}/{len(BOUNDARY_CONFIGS)}]", cfg_key(cfg))
    path = OUT / f"fit-{cfg_key(cfg)}.parquet"

    if path.is_file():
        df = pd.read_parquet(path)
    else:
        df = run_pair_trajectory("PQ32","SQ8",cfg,query_mask=fit_mask)
        df.to_parquet(path,index=False)

    fit_frames.append(df)

fit_boundary = pd.concat(fit_frames,ignore_index=True)
fit_slopes = slopes_from_trajectory(fit_boundary)

print("FIT BOUNDARY SWEEP — COMPLETE")


## 10. Initial-state features and interpretable boundary model


In [ ]:
pq = baseline_cache["PQ32"]
sq = baseline_cache["SQ8"]

features_df = pd.DataFrame({
    "query_id": dev_ids,
    "initial_candidate_divergence": jaccard_distance_rows(pq["ids"],sq["ids"]),
    "initial_abs_utility_gap": np.abs(sq["ndcg"]-pq["ndcg"]),
    "pq32_entropy20": pq["entropy20"],
    "sq8_entropy20": sq["entropy20"],
    "pq32_margin1_10": pq["margin1_10"],
    "sq8_margin1_10": sq["margin1_10"],
    "entropy_gap": np.abs(pq["entropy20"]-sq["entropy20"]),
    "margin_gap": np.abs(pq["margin1_10"]-sq["margin1_10"]),
})

fit_model_df = fit_slopes.merge(features_df,on="query_id",how="left",validate="many_to_one")
fit_model_df["is_softmax"] = (fit_model_df["method"]=="softmax").astype(float)
fit_model_df["temperature_numeric"] = pd.to_numeric(fit_model_df["temperature"],errors="coerce").fillna(1.0)

FEATURES = [
    "initial_candidate_divergence",
    "initial_abs_utility_gap",
    "pq32_entropy20",
    "sq8_entropy20",
    "pq32_margin1_10",
    "sq8_margin1_10",
    "entropy_gap",
    "margin_gap",
    "alpha",
    "k",
    "is_softmax",
    "temperature_numeric",
]

TARGET = "H3_slope"

X_fit = fit_model_df[FEATURES].to_numpy(np.float64)
y_fit = fit_model_df[TARGET].to_numpy(np.float64)

ridge = Pipeline([
    ("scale",StandardScaler()),
    ("model",Ridge(alpha=10.0)),
])
ridge.fit(X_fit,y_fit)

pred_fit = ridge.predict(X_fit)
print("Fit R2:",r2_score(y_fit,pred_fit))
print("Fit MAE:",mean_absolute_error(y_fit,pred_fit))

high_threshold = float(np.quantile(y_fit,0.75))
y_fit_high = (y_fit>=high_threshold).astype(int)

clf = Pipeline([
    ("scale",StandardScaler()),
    ("model",LogisticRegression(C=0.5,max_iter=2000,class_weight="balanced")),
])
clf.fit(X_fit,y_fit_high)

print("Fit AUC:",roc_auc_score(y_fit_high,clf.predict_proba(X_fit)[:,1]))
print("Frozen high-amplification threshold:",high_threshold)


## 11. Boundary validation — untouched half


In [ ]:
val_frames = []

for i,cfg in enumerate(BOUNDARY_CONFIGS,1):
    print(f"[VAL {i}/{len(BOUNDARY_CONFIGS)}]",cfg_key(cfg))
    path = OUT / f"validation-{cfg_key(cfg)}.parquet"

    if path.is_file():
        df = pd.read_parquet(path)
    else:
        df = run_pair_trajectory("PQ32","SQ8",cfg,query_mask=val_mask)
        df.to_parquet(path,index=False)

    val_frames.append(df)

val_boundary = pd.concat(val_frames,ignore_index=True)
val_slopes = slopes_from_trajectory(val_boundary)

val_model_df = val_slopes.merge(features_df,on="query_id",how="left",validate="many_to_one")
val_model_df["is_softmax"] = (val_model_df["method"]=="softmax").astype(float)
val_model_df["temperature_numeric"] = pd.to_numeric(val_model_df["temperature"],errors="coerce").fillna(1.0)

X_val = val_model_df[FEATURES].to_numpy(np.float64)
y_val = val_model_df[TARGET].to_numpy(np.float64)

pred_val = ridge.predict(X_val)
y_val_high = (y_val>=high_threshold).astype(int)
p_val = clf.predict_proba(X_val)[:,1]
yhat_val = (p_val>=0.5).astype(int)

validation_metrics = {
    "r2":float(r2_score(y_val,pred_val)),
    "mae":float(mean_absolute_error(y_val,pred_val)),
    "auc":float(roc_auc_score(y_val_high,p_val)),
    "average_precision":float(average_precision_score(y_val_high,p_val)),
    "accuracy_at_0p5":float(accuracy_score(y_val_high,yhat_val)),
    "fit_high_amplification_threshold":high_threshold,
}

print(json.dumps(validation_metrics,indent=2))
print("OUT-OF-SAMPLE BOUNDARY VALIDATION — COMPLETE")


## 12. Stable / amplifying / reversal regime map


In [ ]:
EPS = 0.002

def regime(x):
    if x > EPS:
        return "amplifying"
    if x < -EPS:
        return "reversal"
    return "stable_or_null"

val_model_df["regime"] = [regime(x) for x in val_model_df["H3_slope"]]

regime_map = (
    val_model_df
    .groupby(["method","alpha","k","temperature_numeric","regime"],as_index=False)
    .size()
    .pivot_table(
        index=["method","alpha","k","temperature_numeric"],
        columns="regime",
        values="size",
        fill_value=0,
    )
    .reset_index()
)

for col in ["amplifying","stable_or_null","reversal"]:
    if col not in regime_map.columns:
        regime_map[col] = 0

total = regime_map["amplifying"]+regime_map["stable_or_null"]+regime_map["reversal"]
regime_map["fraction_amplifying"] = regime_map["amplifying"]/total
regime_map["fraction_stable_or_null"] = regime_map["stable_or_null"]/total
regime_map["fraction_reversal"] = regime_map["reversal"]/total

display(regime_map.sort_values("fraction_amplifying",ascending=False).head(20))
regime_map.to_csv(OUT/"validated_stability_regime_map.csv",index=False)


## 13. Coefficients and final report


In [ ]:
coef_df = pd.DataFrame({
    "feature":FEATURES,
    "ridge_standardized_coefficient":ridge.named_steps["model"].coef_,
    "logistic_standardized_coefficient":clf.named_steps["model"].coef_[0],
})
coef_df.to_csv(OUT/"boundary_model_coefficients.csv",index=False)
display(coef_df.reindex(coef_df["ridge_standardized_coefficient"].abs().sort_values(ascending=False).index))

dose_overall = (
    dose_summary
    .groupby(["low","high"],as_index=False)[["H1_slope","H2_slope","H3_slope"]]
    .mean()
)

report = {
    "status":"ARC_V09_BOUNDARY_STABILITY_MAP_COMPLETE",
    "protocol_sha256":protocol_sha,
    "source_v08_report":str(V08_REPORT),
    "source_v08_report_sha256":sha256_file(V08_REPORT),
    "fit_queries":int(fit_mask.sum()),
    "validation_queries":int(val_mask.sum()),
    "boundary_grid_config_count":len(BOUNDARY_CONFIGS),
    "fidelity_dose_response":dose_overall.to_dict(orient="records"),
    "prediction_target":TARGET,
    "features":FEATURES,
    "validation_metrics":validation_metrics,
    "regime_threshold_abs_slope":EPS,
    "retain_null_and_reversal_cases":True,
    "test_accessed":False,
    "completed_at_utc":datetime.now(timezone.utc).isoformat(),
}

REPORT = OUT/"report.json"
REPORT.write_text(json.dumps(report,indent=2,default=float),encoding="utf-8")
sha = sha256_file(REPORT)
(OUT/"REPORT_SHA256.txt").write_text(sha+"\n",encoding="utf-8")

val_model_df.to_parquet(OUT/"boundary_validation_query_config_rows.parquet",index=False)
dose_overall.to_csv(OUT/"fidelity_dose_response_overall.csv",index=False)

print("Saved:",OUT)
print("Report SHA:",sha)
display(dose_overall)
print(json.dumps(validation_metrics,indent=2))


## Interpretation

ARC-v0.9 strengthens the Full Paper only if it explains both successes and failures.

Useful outcomes include:

- monotonic fidelity dose response;
- reproducible feedback-gain boundary;
- stable/null/reversal regions;
- out-of-sample prediction above chance.

A weak predictive model is also a valid scientific result: it means the replicated phenomenon is
real, but the tested first-order boundary variables do not yet explain query-level susceptibility.

Do not tune mitigation until the v0.9 boundary variables are frozen.
